# Task 9a — omittability by splicing, with the grammaticality control, at power

**Run all cells.** Everything below is idempotent and resumable: re-running a cell is always safe,
and the expensive scoring phase picks up where it left off after a runtime disconnect.

The question. Act 7 of the teaching doc reads two *close* endpoint states as "the text between them
can be skipped". The test: delete whole sentences between two endpoints and ask whether what
*follows* still goes the same way. A **cut** is a pair of token positions `(i, j)`; the spliced
sequence is `ids[:i+1] + ids[j+1:]`. Cuts are **admissible by construction**: both endpoints are
sentence-final (so a sentence end is joined to a sentence start), and at least `WINDOW` tokens
follow `j`. **Divergence** (pre-registered, in the cache header before any cut is scored) is the
median L2 distance between the spliced text's next-token log-probability vector at `i+1+k` and the
original's at `j+1+k`, `k = 0..WINDOW-1`. It is deliberately *not* Act 3's fluency.

Two controls the earlier version lacked. (1) **Length matching is mandatory**: divergence tracks how
much text was deleted (Spearman ≈ +0.9 on the demo paragraph) and so does endpoint distance, so
close / far are the bottom / top decile of endpoint distance **within length-matching bins** inside
each deleted-length stratum, and the estimate is stratified. A pooled ratio above 1 is not a pass.
(2) **A hand audit** of 30 admissible cuts (10 close, 10 far, 10 other), judged on text alone,
records whether the rule held; failures are reported and excluded, never dropped silently.

Admissibility is checked at **both** ends: the head must end a sentence *and* the tail must resume at
one. The right-side half of that rule exists because the audit of an earlier run found endpoints that
were mid-sentence — a `Vol.` that punkt split on, a dropped wiki template leaving `, 39 volumes …`.
Such a position is mid-sentence, so its state is atypical for a sentence end and lands in the **far**
decile far more often than the close one (13.5% vs 0.5% in that run). The defect correlated with the
exposure variable, so it is now rejected by construction; `diagnostics.boundary_check` re-verifies it
on every run and the verdict warns if any survive.

What this notebook does:

1. Clones the private repo `SalmonSung/m1_llms_analyzer` and installs its dependencies.
2. Runs a **smoke test** of the whole experiment on a ~5 MB model and three hand paragraphs.
3. Loads the model with its LM head and streams **N Wikipedia paragraphs** of 150–300 tokens.
4. Walks through **one paragraph**: boundaries, admissible cuts, one splice.
5. **Phase A (GPU):** scores every admissible cut of every paragraph into a resumable JSONL cache.
6. Exports the **audit sheet**; you fill in `grammatical` (y/n) and it is read back.
7. **Phase B (CPU):** labels, strata, the stratified permutation test, the paragraph-level cluster
   bootstrap, the pooled and continuous diagnostics; the `fig_9a` record and figure.

## 1 · Bootstrap — clone the repo

In [ ]:
#@title Clone (or update) the repository { display-mode: "form" }
import base64, json, os, subprocess, sys, textwrap, urllib.error, urllib.request
from pathlib import Path

REPO_OWNER  = "SalmonSung"
REPO_NAME   = "m1_llms_analyzer"
REPO_BRANCH = "main"   #@param {type:"string"}

CLEAN_URL = f"https://github.com/{REPO_OWNER}/{REPO_NAME}.git"
NEW_PAT_URL = "https://github.com/settings/personal-access-tokens/new"


def _in_colab() -> bool:
    try:
        import google.colab  # noqa: F401
        return True
    except Exception:
        return False


def _colab_secret(name):
    """Read a Colab secret, returning None if it is absent or access is denied."""
    if not _in_colab():
        return None
    try:
        from google.colab import userdata
        return userdata.get(name) or None
    except Exception as exc:
        print(f"  (Colab secret {name!r} unavailable: {type(exc).__name__})")
        return None


def _clean(token):
    """Strip whitespace and stray quotes -- by far the most common paste error."""
    if not token:
        return None
    return token.strip().strip('"').strip("'").strip() or None


def _token_kind(token):
    """Name the token type from its prefix, without revealing the value."""
    for prefix, kind in (
        ("github_pat_", "fine-grained PAT"),
        ("ghp_", "classic PAT"),
        ("gho_", "OAuth token"),
        ("ghs_", "App installation token"),
        ("ghu_", "user-to-server token"),
    ):
        if token.startswith(prefix):
            return kind
    return "UNRECOGNISED PREFIX"


def _api(path, token):
    """GET api.github.com/<path> with the token. Raises urllib.error.HTTPError."""
    request = urllib.request.Request(
        f"https://api.github.com/{path}",
        headers={
            "Authorization": f"Bearer {token}",
            "Accept": "application/vnd.github+json",
            "X-GitHub-Api-Version": "2022-11-28",
        },
    )
    with urllib.request.urlopen(request, timeout=30) as response:
        return json.loads(response.read().decode())


def _auth_config(token):
    """Auth as a per-command git config value, so it never touches .git/config or a URL."""
    basic = base64.b64encode(f"x-access-token:{token}".encode()).decode()
    return f"http.extraHeader=AUTHORIZATION: basic {basic}"


def _run(cmd, token=None):
    """Run git, redacting the auth header and the token from anything printed."""
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        message = result.stderr or result.stdout
        if token:
            message = message.replace(token, "***")
        shown = " ".join("<auth>" if "extraHeader" in c else c for c in cmd)
        raise RuntimeError(f"git failed: {shown}\n{message}")
    return result.stdout.strip()


def _preflight(token):
    """Verify the token before git runs, so failures name their actual cause.

    A bare `git clone` failure says only 'Invalid username or token', which covers an
    expired token, a typo, a missing repo grant, and un-authorised SSO alike. These two
    API calls tell those apart.
    """
    kind = _token_kind(token)
    print(f"GITHUB_TOKEN: {len(token)} chars, looks like a {kind}.")
    if kind == "UNRECOGNISED PREFIX":
        print("  Warning: GitHub tokens start with github_pat_, ghp_, gho_, ghs_ or ghu_.")
        print("  If you pasted an account password or an SSH key, that will not work here.")

    try:
        me = _api("user", token)
    except urllib.error.HTTPError as exc:
        if exc.code == 401:
            raise SystemExit(textwrap.dedent(f"""
                GitHub rejected this token (401 Unauthorized). The token itself is bad --
                this is not a permissions problem. Most likely one of:

                  * it has expired (fine-grained PATs expire, 30 days by default);
                  * it was revoked or regenerated;
                  * the secret holds something that is not a token (an account password
                    will never work -- GitHub removed password auth for git);
                  * it was truncated or mangled when pasted.

                Fix: create a new token at
                  {NEW_PAT_URL}
                  - Resource owner: {REPO_OWNER}
                  - Repository access: only select repositories -> {REPO_NAME}
                  - Permissions: Repository permissions -> Contents -> Read-only
                Then in Colab: key icon in the left sidebar -> edit GITHUB_TOKEN, paste the
                new value with no quotes and no trailing spaces, keep 'Notebook access' on,
                and re-run this cell.
            """).strip())
        if exc.code == 403:
            raise SystemExit(textwrap.dedent(f"""
                GitHub returned 403 for this token. Usually either a rate limit, or the
                token needs SAML SSO authorisation for the '{REPO_OWNER}' organisation.
                If {REPO_OWNER} is an org with SSO, open your token's settings page and
                click 'Configure SSO' -> Authorize.

                Original error: {exc}
            """).strip())
        raise

    print(f"  Authenticates as: {me.get('login')}")

    try:
        repo = _api(f"repos/{REPO_OWNER}/{REPO_NAME}", token)
    except urllib.error.HTTPError as exc:
        if exc.code == 404:
            login = me.get("login")
            not_owner = (
                f"\n                  * you are {login}, but the repo belongs to "
                f"{REPO_OWNER} and you are not a collaborator on it;"
                if login and login.lower() != REPO_OWNER.lower()
                else ""
            )
            raise SystemExit(textwrap.dedent(f"""
                The token is valid (you are {login}), but it cannot see
                {REPO_OWNER}/{REPO_NAME}. GitHub returns 404 rather than 403 for a private
                repo a token has no grant on, so this means one of:

                  * the token's 'Repository access' does not include {REPO_NAME};
                  * it lacks the 'Contents: Read-only' repository permission;{not_owner}
                  * the owner or name is misspelled (both are case-sensitive).

                Fix: open {NEW_PAT_URL} (or edit the existing token), grant this
                repository and Contents: Read-only, then re-run this cell.
            """).strip())
        raise

    print(f"  Repo access:      OK ({'private' if repo.get('private') else 'public'})")
    return True


_TOKEN = _clean(_colab_secret("GITHUB_TOKEN") or os.environ.get("GITHUB_TOKEN"))

if not _in_colab() and Path("pyproject.toml").exists():
    # Running from a local checkout -- nothing to clone.
    REPO_DIR = Path.cwd()
    print(f"Local checkout detected: {REPO_DIR}")
else:
    REPO_DIR = Path("/content") / REPO_NAME if _in_colab() else Path.cwd() / REPO_NAME
    if not _TOKEN:
        raise SystemExit(textwrap.dedent(f"""
            GITHUB_TOKEN is not set, and {REPO_OWNER}/{REPO_NAME} is private.
              1. Create a fine-grained PAT at {NEW_PAT_URL}
                 - Resource owner: {REPO_OWNER}
                 - Repository access: only select repositories -> {REPO_NAME}
                 - Permissions: Contents -> Read-only
              2. Colab left sidebar -> key icon -> add a secret named GITHUB_TOKEN
              3. Turn on 'Notebook access' for it, then re-run this cell.
        """).strip())

    _preflight(_TOKEN)

    # Auth travels as a per-command header, never in the URL. Nothing is written to
    # .git/config, so there is no token left on disk to scrub afterwards.
    _AUTH = _auth_config(_TOKEN)
    try:
        if REPO_DIR.exists():
            print(f"\nRepo already present at {REPO_DIR}; updating...")
            _run(["git", "-C", str(REPO_DIR), "remote", "set-url", "origin", CLEAN_URL], _TOKEN)
            _run(["git", "-C", str(REPO_DIR), "-c", _AUTH, "fetch", "origin", REPO_BRANCH], _TOKEN)
            _run(["git", "-C", str(REPO_DIR), "checkout", REPO_BRANCH], _TOKEN)
            _run(["git", "-C", str(REPO_DIR), "reset", "--hard", f"origin/{REPO_BRANCH}"], _TOKEN)
        else:
            print(f"\nCloning into {REPO_DIR} ...")
            _run(["git", "-c", _AUTH, "clone", "--branch", REPO_BRANCH, "--depth", "1",
                  CLEAN_URL, str(REPO_DIR)], _TOKEN)
    except RuntimeError as exc:
        # The API accepted the token but git did not -- rare, and worth naming, because
        # the obvious readings (bad token, missing grant) were just ruled out above.
        raise SystemExit(textwrap.dedent(f"""
            {exc}

            The token passed the API preflight above, so it is valid and can see this
            repo -- the failure is in the git transport itself. Things to check:

              * branch '{REPO_BRANCH}' exists on the remote (a typo in REPO_BRANCH gives
                'Remote branch not found');
              * a stale {REPO_DIR} from an earlier run: delete it and re-run this cell;
              * a corporate proxy or VPN intercepting HTTPS to github.com.
        """).strip())
    print("Done. Remote is", CLEAN_URL, "(no credentials stored on disk).")

os.chdir(REPO_DIR)
print("Working directory:", Path.cwd())
print("Commit:", _run(["git", "rev-parse", "--short", "HEAD"]))
del _TOKEN  # do not leave the token bound in the notebook namespace

## 2 · Dependencies

Installed with `--upgrade-strategy only-if-needed` so Colab's preinstalled,
CUDA-matched `torch` is **kept** rather than reinstalled (a torch swap costs several
minutes and can break GPU support).

In [ ]:
#@title Install dependencies
import subprocess, sys

print("Installing (quiet; ~30s on a cold runtime)...")
result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "--upgrade-strategy", "only-if-needed", "-e", "."],
    capture_output=True, text=True,
)
print(result.stdout[-2000:] or "(no output)")
if result.returncode != 0:
    print(result.stderr[-3000:], file=sys.stderr)
    raise SystemExit("Dependency installation failed -- see the error above.")

# Make the freshly installed package importable in this already-running kernel.
import importlib, site
importlib.reload(site)
for module in [m for m in list(sys.modules) if m.startswith("m1_analyzer")]:
    del sys.modules[module]

import m1_analyzer
print("m1_analyzer", m1_analyzer.__version__, "ready")

## 3 · Environment report

In [ ]:
#@title What am I running on?
import torch, transformers, numpy, platform
from m1_analyzer import in_colab, resolve_hf_token

print(f"python        : {platform.python_version()}")
print(f"torch         : {torch.__version__}")
print(f"transformers  : {transformers.__version__}")
print(f"numpy         : {numpy.__version__}")
print(f"in Colab      : {in_colab()}")

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"GPU           : {props.name} ({props.total_memory / 1024**3:.1f} GB)")
    print(f"CUDA          : {torch.version.cuda}")
else:
    print("GPU           : none -- running on CPU.")
    print("                Runtime -> Change runtime type -> T4 GPU for anything above ~1B params.")

# Only reports presence. The token value is never printed.
print(f"HF_TOKEN      : {'found' if resolve_hf_token() else 'not set (fine for ungated models)'}")

# The dtype the scorer will run in. A T4 gets float16 (its bfloat16 is emulated and slow);
# Ampere and newer get bfloat16; CPU gets float32.
from m1_analyzer.utils.device import resolve_device, resolve_dtype
print(f"scoring dtype : {str(resolve_dtype(resolve_device('auto'), 'auto')).replace('torch.', '')}")

## 4 · Configuration

**This is the only cell you normally edit.**

| Setting | Meaning |
|---|---|
| `MODEL_ID` | Any decoder-only Hugging Face model id. Use a **base** model. Raw L2 distances scale with the vocabulary size, so numbers are comparable within one model only. |
| `DTYPE` | `auto` picks fp16 on a T4. Switch to `float32` if phase A reports a non-finite state. |
| `CORPUS` | `wikipedia` streams `wikimedia/wikipedia` (untokenised prose, real paragraph breaks); or a path to a `.txt` (blank-line-separated paragraphs) / `.jsonl` (`text` field). |
| `N_PARAGRAPHS`, `MIN_TOKENS`, `MAX_TOKENS`, `MIN_SENTENCES` | The corpus filter, in the model's own tokens. ~200 paragraphs of 150–300 tokens give a few thousand admissible cuts. |
| `WINDOW` | Aligned positions after the rejoin that the divergence is taken over. **Pre-registered**: fixed in the cache header before scoring; a different value refuses to resume. |
| `STRATA`, `DECILE`, `MATCH_WIDTH` | Deleted-length strata `[lo, hi)`, the close/far decile, and the width of the length-matching bins the deciles are taken within. All pre-registered in the header. |
| `SPLITTER` | `punkt` (NLTK, knows `Mr.` and `U.S.`) for real text; `regex` only for offline tests. |
| `BOUNDARY_KINDS` | `sentence` is the primary type; `clause` (`, ; :` and dashes) is generated and stored but analysed separately, since comma-to-comma joins are not grammatical by construction. |
| `AUDIT_*` | How many close / far / other cuts the audit sheet holds. |
| `BATCH_SIZE` | Spliced sequences per forward pass; halves on OOM. |

In [ ]:
#@title Run configuration { display-mode: "form" }
SMOKE_MODEL_ID  = "sshleifer/tiny-gpt2"        #@param {type:"string"}
MODEL_ID        = "Qwen/Qwen3-0.6B-Base"       #@param {type:"string"}
DTYPE           = "auto"                       #@param ["auto", "float16", "bfloat16", "float32"]
CORPUS          = "wikipedia"                  #@param {type:"string"}
N_PARAGRAPHS    = 200                          #@param {type:"integer"}
MIN_TOKENS      = 150                          #@param {type:"integer"}
MAX_TOKENS      = 300                          #@param {type:"integer"}
MIN_SENTENCES   = 5                            #@param {type:"integer"}
MAX_ARTICLES    = 20000                        #@param {type:"integer"}
WINDOW          = 20                           #@param {type:"integer"}
STRATA          = "20-40,40-80,80-160"         #@param {type:"string"}
DECILE          = 0.1                          #@param {type:"number"}
MATCH_WIDTH     = 10                           #@param {type:"integer"}
SPLITTER        = "punkt"                      #@param ["punkt", "regex"]
BOUNDARY_KINDS  = "sentence,clause"            #@param {type:"string"}
AUDIT_CLOSE     = 10                           #@param {type:"integer"}
AUDIT_FAR       = 10                           #@param {type:"integer"}
AUDIT_OTHER     = 10                           #@param {type:"integer"}
N_PERM          = 5000                         #@param {type:"integer"}
N_BOOT          = 1000                         #@param {type:"integer"}
MIN_PER_GROUP   = 10                           #@param {type:"integer"}
BATCH_SIZE      = 4                            #@param {type:"integer"}
OUTPUT_DIR      = "outputs/task_9a"            #@param {type:"string"}
SEED            = 42                           #@param {type:"integer"}
RESUME          = True                         #@param {type:"boolean"}
MIRROR_TO_DRIVE = False                        #@param {type:"boolean"}
DRIVE_DIR       = "/content/drive/MyDrive/m1_llms_analyzer/task_9a"  #@param {type:"string"}

import re
from pathlib import Path

from m1_analyzer import ModelConfig, RunConfig, ScoringConfig, StorageConfig

OUT = Path(OUTPUT_DIR)
OUT.mkdir(parents=True, exist_ok=True)
STRATA_EDGES = [[int(a), int(b)] for a, b in (s.split("-") for s in STRATA.split(","))]
KINDS = tuple(k.strip() for k in BOUNDARY_KINDS.split(",") if k.strip())


def slug(text: str) -> str:
    """Filesystem-safe name (model id, corpus) for output files."""
    return re.sub(r"-{2,}", "-", re.sub(r"[^A-Za-z0-9._-]+", "-", text)).strip("-")


def make_config(model_id: str, **overrides) -> RunConfig:
    """A RunConfig with the LM head, so next-token states and log-probabilities work."""
    return RunConfig(
        model=ModelConfig(model_id=model_id, head="causal_lm", dtype=overrides.get("dtype", DTYPE)),
        scoring=ScoringConfig(batch_size=16, max_length_cap=max(512, MAX_TOKENS + 8)),
        storage=StorageConfig(output_dir=str(OUT)),
        seed=SEED,
    )


# Every output is keyed by model AND corpus, so two runs never clash on a cache header.
RUN_TAG = f"{slug(MODEL_ID)}_{slug(Path(CORPUS).stem if CORPUS != 'wikipedia' else 'wikipedia')}_w{WINDOW}"
CORPUS_PATH = OUT / f"paragraphs_{RUN_TAG}.jsonl"
CACHE_PATH = OUT / f"splices_{RUN_TAG}.jsonl"
AUDIT_PATH = OUT / f"audit_{RUN_TAG}.csv"
RECORD_PATH = OUT / f"record_9a_{RUN_TAG}.json"
FIG_PATH = OUT / f"fig_9a_{RUN_TAG}.png"

print(f"strata   : {STRATA_EDGES}  decile {DECILE}  matching bins {MATCH_WIDTH} tokens  window {WINDOW}")
print(f"cache    : {CACHE_PATH}")
print(f"audit    : {AUDIT_PATH}")
print("Configuration ready.")

## 5 · Smoke test (~5 MB, a few seconds)

Runs the *entire* experiment — boundaries, cuts, states, cache, labels, the stratified test, the
audit sheet, the figure — on a tiny model and three hand paragraphs, with the `regex` splitter and a
short window so the tiny texts have cuts. If this passes, the only things that can still go wrong
with the real model are download size, GPU memory, and the corpus stream.

In [ ]:
#@title Smoke test on a tiny model
import time

import matplotlib
matplotlib.use("Agg")

from m1_analyzer import Analyzer
from m1_analyzer.experiments import experiment_figures as EF
from m1_analyzer.experiments import (
    analyse_9a, assign_pairs, audit_sample, compute_splices, cut_count, flatten_cuts, hand_paragraphs,
    validate_record_9a, verdict_9a, write_audit_csv,
)

smoke = Analyzer(make_config(SMOKE_MODEL_ID))
SMOKE_CACHE = OUT / "smoke_splices.jsonl"
if SMOKE_CACHE.exists():
    SMOKE_CACHE.unlink()
print(smoke.describe()["head"], "|", smoke.models.architecture, "| BOS token:", smoke.states.bos_token())

started = time.perf_counter()
header, rows = compute_splices(
    smoke.states, hand_paragraphs(), window=5, splitter="regex", cache_path=SMOKE_CACHE,
    strata=[[1, 10], [10, 30], [30, 80]], provenance={"model_id": SMOKE_MODEL_ID}, show_progress=False,
)
cuts = flatten_cuts(rows)
assign_pairs(cuts, strata=header["strata"], decile=header["decile"], match_width=header["match_width"])
write_audit_csv(audit_sample(cuts, n_close=2, n_far=2, n_other=2, seed=SEED), rows, OUT / "smoke_audit.csv")
smoke_record = analyse_9a(rows, header, model=SMOKE_MODEL_ID, n_perm=200, n_boot=100, min_per_group=1)
validate_record_9a(smoke_record)
print(f"\n{len(rows)} paragraphs, {cut_count(rows)} admissible cuts scored and analysed in {time.perf_counter() - started:.1f}s")
print(verdict_9a(smoke_record))
EF.fig_9a(smoke_record, path=str(OUT / "smoke_fig_9a.png"))
assert (OUT / "smoke_fig_9a.png").stat().st_size > 10_000
print("\nSmoke test passed (the numbers above are meaningless: a random 5 MB model).")

smoke.unload()   # free the tiny model before loading the real one

## 6 · Load the model

First download takes a minute; it is cached for the rest of the session. The model is loaded
**with its language-model head** (`head="causal_lm"`): that is what turns a position into a
next-token log-probability vector. If it is gated, accept its licence on the Hub and add
`HF_TOKEN` to Colab secrets.

In [ ]:
#@title Load the model
import time

from m1_analyzer import Analyzer

started = time.perf_counter()
analyzer = Analyzer(make_config(MODEL_ID))
print(f"Loaded in {time.perf_counter() - started:.1f}s\n")
for key, value in analyzer.describe().items():
    if key != "layers":
        print(f"{key:>18} : {value}")
print(f"{'BOS token':>18} : {analyzer.states.bos_token()!r} (prepended so position 0 has a state)")
print(f"{'vocabulary':>18} : {analyzer.states.vocab_size} (the dimension every distance lives in)")

count_tokens = lambda text: len(analyzer.states.encode(text))
PROVENANCE = {
    "model_id": MODEL_ID,
    "revision": analyzer.models.metadata()["revision"],
    "dtype": analyzer.models.metadata()["dtype"],
    "bos_token": analyzer.states.bos_token(),
    "vocab_size": analyzer.states.vocab_size,
    "seed": SEED,
}

## 7 · The corpus

Paragraphs of `MIN_TOKENS`–`MAX_TOKENS` tokens (in *this* model's tokens) with at least
`MIN_SENTENCES` sentences, taken in source order into a pool and sampled with `SEED`. The
selection is written to a JSONL next to the cache, so a re-run reads the same paragraphs back
instead of streaming again. With `SPLITTER = "punkt"` the NLTK sentence model is downloaded once.

In [ ]:
#@title Load N paragraphs
import collections
import json
import time

from m1_analyzer.experiments import (
    Paragraph, WIKIPEDIA_NAME, load_paragraph_file, load_wikipedia_paragraphs, select_paragraphs,
)

if SPLITTER == "punkt":
    import nltk
    nltk.download("punkt_tab", quiet=True)

started = time.perf_counter()
if RESUME and CORPUS_PATH.exists():
    with CORPUS_PATH.open(encoding="utf-8") as fh:
        paragraphs = [Paragraph.from_json(json.loads(line)) for line in fh if line.strip()]
    print(f"Read {len(paragraphs)} paragraphs back from {CORPUS_PATH}")
else:
    kwargs = dict(n=N_PARAGRAPHS, min_tokens=MIN_TOKENS, max_tokens=MAX_TOKENS,
                  min_sentences=MIN_SENTENCES, splitter=SPLITTER, seed=SEED)
    if CORPUS == "wikipedia":
        paragraphs = load_wikipedia_paragraphs(count_tokens, max_articles=MAX_ARTICLES, **kwargs)
    else:
        paragraphs = load_paragraph_file(CORPUS, count_tokens, **kwargs)
    with CORPUS_PATH.open("w", encoding="utf-8") as fh:
        for p in paragraphs:
            fh.write(json.dumps(p.to_json(), ensure_ascii=False) + "\n")
    print(f"Selected {len(paragraphs)} paragraphs from {paragraphs[0].source} in {time.perf_counter() - started:.1f}s")

CORPUS_NAME = paragraphs[0].source
PROVENANCE["corpus"] = CORPUS_NAME
tokens = [p.info["n_tokens"] for p in paragraphs]
sents = [p.info["n_sentences"] for p in paragraphs]
print(f"tokens    : min {min(tokens)}  median {sorted(tokens)[len(tokens) // 2]}  max {max(tokens)}")
print(f"sentences : {dict(sorted(collections.Counter(sents).items()))}")
print(f"\n{paragraphs[0].id}: {paragraphs[0].text[:300]}...")

## 8 · One paragraph, end to end

Boundaries, the admissible cuts they generate, and one splice shown as text. Sentence-final
endpoints only join a sentence end to a sentence start, so no cut here lands mid-word or
mid-clause — that is the confound the earlier version had (its far pair left `'The' + 'icates any
simple story'`), replaced here by construction rather than by labelling.

In [ ]:
#@title Walk through one paragraph
from m1_analyzer.experiments import SENTENCE, admissible_cuts, paragraph_boundaries, score_paragraph, splice_text

one = paragraphs[0]
ids, offsets, boundaries, rejected = paragraph_boundaries(analyzer.states, one, splitter=SPLITTER, boundary_kinds=KINDS)
print(f"{len(ids)} tokens; boundaries: " + ", ".join(f"{k} {len(v)}" for k, v in boundaries.items())
      + f"; {rejected} sentence end(s) rejected by the rule")
cuts = admissible_cuts(boundaries, len(ids), WINDOW)
print(f"{len(cuts)} admissible cuts ({sum(c.boundary == SENTENCE for c in cuts)} sentence-final)\n")

row = score_paragraph(analyzer.states, one, window=WINDOW, splitter=SPLITTER, boundary_kinds=KINDS, batch_size=BATCH_SIZE)
sent = sorted((c for c in row["cuts"] if c["boundary"] == SENTENCE), key=lambda c: c["d"])
if not sent:
    raise SystemExit("This paragraph has no admissible sentence cut. If `rejected` above is most of the "
                     "sentence ends, the boundary rule disagrees with this tokenizer's offsets; "
                     "otherwise pick another paragraph (paragraphs[1], ...).")
for label, c in (("closest endpoints", sent[0]), ("farthest endpoints", sent[-1])):
    print(f"{label}: positions {c['i']}->{c['j']}, {c['seg_len']} tokens deleted, endpoint distance {c['d']:.1f}, "
          f"divergence {c['div']:.1f}, fluency {c['fl']:.3f} vs {row['fluency']:.3f} nats/token, "
          f"re-tokenises: {c['retokenises']}")
    print("   " + splice_text(one.text, offsets, c["i"], c["j"], marker=" ⟦cut⟧ ")[:400] + "\n")

## 9 · Phase A — score every admissible cut (GPU)

For each paragraph: one forward pass of the original text gives every endpoint state and every
downstream state; one forward pass per cut gives the spliced states at the rejoin. Splicing is done
on token ids, so spliced position `i+1+k` and original position `j+1+k` are the same token by
construction. One JSONL line per finished paragraph, fsynced, so a disconnect costs at most one
paragraph: re-run this cell and it resumes. The **header line is the pre-registration** (measure,
window, boundary kinds, strata, decile, matching width): a run with different settings refuses to
resume and names the field.

In [ ]:
#@title Score all cuts (resumable)
import shutil
import time

from m1_analyzer.experiments import compute_splices, cut_count

if not RESUME and CACHE_PATH.exists():
    CACHE_PATH.unlink()
    print("RESUME=False: deleted the existing cache; scoring from scratch.")

mirror = None
if MIRROR_TO_DRIVE:
    from google.colab import drive  # noqa: F401 -- Colab only
    drive.mount("/content/drive")
    mirror = Path(DRIVE_DIR) / CACHE_PATH.name
    if not CACHE_PATH.exists() and mirror.exists():
        shutil.copy(mirror, CACHE_PATH)
        print(f"Restored cache from Drive: {mirror}")

started = time.perf_counter()
header, rows = compute_splices(
    analyzer.states, paragraphs, window=WINDOW, cache_path=CACHE_PATH, provenance=PROVENANCE,
    splitter=SPLITTER, boundary_kinds=KINDS, primary_boundary="sentence", strata=STRATA_EDGES,
    decile=DECILE, match_width=MATCH_WIDTH, batch_size=BATCH_SIZE, show_progress=True,
    mirror_path=mirror,
)
print(f"\n{len(rows)} paragraphs scored in {(time.perf_counter() - started) / 60:.1f} min")
for kind in KINDS:
    print(f"  {kind:<9} cuts: {cut_count(rows, kind)}")
print(f"  sentence ends rejected by the rule: {sum(r['n_rejected_boundaries'] for r in rows)}")
print(f"  cuts whose decoded text does not re-tokenise to the same ids: "
      f"{sum(1 for r in rows for c in r['cuts'] if not c['retokenises'])}")

## 9b · Deleted-segment surprisal (additive, one pass per paragraph)

The length-matched comparison answers one objection and leaves another: far pairs may simply delete
more *information* per token, so "more changed" would be trivially true. To test that, every cut
needs one more number — how much information the deleted span carried. From **the same
original-text pass** that produced the endpoint and downstream states (no splicing), take the
log-probability of the token that actually came next:

```
surprisal[t] = -log p(ids[t] | ids[:t])        nats; float32 log-softmax, token 0 scored from the BOS row
del_surp      = sum(surprisal[i+1 : j+1])       the deleted tokens i+1 .. j, seg_len of them
del_surp_mean = del_surp / seg_len
```

`surprisal[t]` is read from the state at position `t-1` — the same vector `d` and `div` use there.
This cell fills `surprisal` (per paragraph) and `del_surp` / `del_surp_mean` (per cut) into the
cache **additively**: rows that already carry them are skipped, every other key and value is
re-serialised byte for byte, the header gains `surprisal_measure`, the file is rewritten atomically
(and mirrored back to Drive when `MIRROR_TO_DRIVE` is on). `d`, `div`, `div_k`, `fl`, `i`, `j` do not
move, so a filled audit sheet and an existing record still line up. Running it on a cache scored
after this field was added is a no-op.

It also prints the two invariants: every paragraph's `surprisal` is `n_tokens` long and every cut's
deleted span is `seg_len` long; and `mean(surprisal)` reproduces the paragraph's `fluency`.
**`fluency` is the mean over all tokens, token 0 included** (it is scored from the BOS row), so
`mean(surprisal)` matches to rounding and `mean(surprisal[1:])` does not — that difference is the
BOS convention, not an indexing error, and nothing is adjusted for it.

Needs the model (cell 6) and `CACHE_PATH` (cell 4); in a fresh runtime it restores the cache from
`DRIVE_DIR` first. Phase B below then reads the augmented rows.


In [ ]:
#@title Add surprisal to the cache (additive, resumable)
import shutil
from pathlib import Path

from m1_analyzer import in_colab
from m1_analyzer.experiments import add_surprisal
from m1_analyzer.experiments.jsonl_cache import mirror as mirror_cache

if "analyzer" not in globals():
    raise SystemExit("Run '6 · Load the model' first: the surprisal pass needs the model that scored the cache.")

DRIVE_CACHE = Path(DRIVE_DIR) / CACHE_PATH.name
if MIRROR_TO_DRIVE and in_colab():
    from google.colab import drive  # noqa: F401 -- Colab only
    drive.mount("/content/drive")
    if not CACHE_PATH.exists() and DRIVE_CACHE.exists():
        CACHE_PATH.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy(DRIVE_CACHE, CACHE_PATH)
        print(f"Restored the cache from Drive: {DRIVE_CACHE}")
if not CACHE_PATH.exists():
    raise SystemExit(f"No cache at {CACHE_PATH} (and none at {DRIVE_CACHE}). Run Phase A first, or turn on "
                     "MIRROR_TO_DRIVE with DRIVE_DIR pointing at the mirrored splices JSONL.")

header, rows, report = add_surprisal(analyzer.states, CACHE_PATH, batch_size=1, show_progress=True)
n = report["n_with_surprisal"]
print(f"\n{report['n_augmented']} paragraph(s) augmented, {report['n_skipped']} already had surprisal; "
      f"written: {report['written'] or 'nothing (no change)'}")
print(f"invariant 1  lengths : {report['n_bad_paragraph_length']}/{n} paragraphs with len(surprisal) != n_tokens; "
      f"{report['n_bad_cut_length']}/{report['n_cuts']} cuts with len(surprisal[i+1:j+1]) != seg_len")
print(f"invariant 2  fluency : max |mean(surprisal)     - fluency| = {report['max_abs_mean_all_minus_fluency']:.2e}  "
      f"({report['n_within_tol_all']}/{n} within {report['tol']})")
print(f"                       max |mean(surprisal[1:]) - fluency| = {report['max_abs_mean_from_1_minus_fluency']:.2e}  "
      f"({report['n_within_tol_from_1']}/{n} within {report['tol']})")
print(f"             del_surp: max |del_surp - sum(surprisal[i+1:j+1])| = {report['max_abs_del_surp_minus_sum']:.1e}")
print("definition   :", report["fluency_definition"])
if not report["ok"]:
    raise SystemExit("An invariant failed -- see the lines above. Nothing was adjusted; report the numbers as they are.")

if MIRROR_TO_DRIVE and in_colab() and report["written"]:
    mirror_cache(CACHE_PATH, DRIVE_CACHE)
    print("mirrored back to", DRIVE_CACHE)


## 10 · The audit sheet (you, by hand)

Admissibility makes the splice grammatical *by construction*, so the `grammatical` field records
whether that held when a person reads it.

**This step is still required, and is not made redundant by the tightened boundary rule.** The audit
is what *found* that rule's gap in the first place; an automated check can only test failure modes
someone has already thought of. A changed rule needs fresh evidence that it holds, so audit this run
as if the rule were untested — which, on this corpus, it is. This cell draws `AUDIT_CLOSE` + `AUDIT_FAR` + `AUDIT_OTHER`
admissible sentence-final cuts with `SEED` and writes a CSV showing only **text** — the spliced
paragraph with the join marked, never a divergence — so the judgement is blind.

**Fill in the `grammatical` column with `y` or `n`** (and a note for any `n`), keeping the file
name, then put it back at the path printed below: upload it in the next cell, or drop it into the
Drive folder if `MIRROR_TO_DRIVE` is on. An existing sheet is never overwritten.

In [ ]:
#@title Export the audit sheet
from m1_analyzer.experiments import assign_pairs, audit_sample, flatten_cuts, write_audit_csv

cuts = flatten_cuts(rows)
assign_pairs(cuts, strata=header["strata"], decile=header["decile"], match_width=header["match_width"],
             boundary=header["primary_boundary"])
sample = audit_sample(cuts, n_close=AUDIT_CLOSE, n_far=AUDIT_FAR, n_other=AUDIT_OTHER, seed=SEED,
                      boundary=header["primary_boundary"])

if AUDIT_PATH.exists():
    print(f"Audit sheet already exists, not overwritten: {AUDIT_PATH}")
else:
    write_audit_csv(sample, rows, AUDIT_PATH)
    print(f"Wrote {len(sample)} cuts to {AUDIT_PATH}")
    if MIRROR_TO_DRIVE:
        shutil.copy(AUDIT_PATH, Path(DRIVE_DIR) / AUDIT_PATH.name)
        print("  (copied to Drive)")

for k, c in enumerate(sample, 1):
    print(f"\n[{k:02d}] {c['key']}  {c['audit_group']}  {c['seg_len']} tokens deleted")
    print("     ..." + c["join"] + "...")

from m1_analyzer import in_colab
if in_colab():
    from google.colab import files
    files.download(str(AUDIT_PATH))

## 11 · Phase B — labels, strata, the stratified test (CPU)

Reads the cache and the filled audit sheet. Close / far are the bottom / top `DECILE` of endpoint
distance within each `MATCH_WIDTH`-token bin of deleted length, pooled across paragraphs; the
estimate is the stratum-size-weighted far/close median ratio over the pre-registered strata; the
p-value is a permutation test that shuffles labels within bins; the interval is a paragraph-level
cluster bootstrap. The pooled comparison and the residual (continuous) analysis are diagnostics.
If the audit sheet is absent or unfilled, the analysis still runs with `audited = false`
everywhere and the verdict says so.

**This phase needs no GPU, no model and no corpus** — only the cache Phase A wrote. So it also runs in a fresh runtime days later: if `rows` is not in memory, the cell copies `splices_<RUN_TAG>.jsonl` (and the audit sheet) back from `DRIVE_DIR` and reads `header, rows` from it, taking the model, dtype, BOS token and corpus from the cache header rather than from this session. Set `LOAD_FROM_DRIVE` to force that restore even when a local cache exists — the Drive copy then wins. `MIRROR_TO_DRIVE` or `LOAD_FROM_DRIVE` mounts Drive; `DRIVE_DIR` and the `MODEL_ID` / `CORPUS` / `WINDOW` settings must be the ones the run was scored under, since `RUN_TAG` is what names the file.

In [ ]:
#@title Analyse
import shutil
from pathlib import Path

from m1_analyzer import in_colab
from m1_analyzer.experiments import analyse_9a, cut_count, load_audit_csv, load_splices, verdict_9a

LOAD_FROM_DRIVE = False  #@param {type:"boolean"}
UPLOAD_AUDIT = False  #@param {type:"boolean"}

# Phase B is CPU-only and reads nothing but the cache, so it can run in a fresh runtime that
# never loaded the model or the corpus: `header, rows` come back from the JSONL. Restore that
# file from the Drive mirror when it is not on local disk (or when LOAD_FROM_DRIVE forces it),
# then read it. With `rows` already in memory from Phase A, nothing here touches Drive.
DRIVE = Path(DRIVE_DIR)
if (MIRROR_TO_DRIVE or LOAD_FROM_DRIVE) and in_colab() and not DRIVE.exists():
    from google.colab import drive  # noqa: F401 -- Colab only
    drive.mount("/content/drive")

if LOAD_FROM_DRIVE or "rows" not in globals() or "header" not in globals():
    if (LOAD_FROM_DRIVE or not CACHE_PATH.exists()) and (DRIVE / CACHE_PATH.name).exists():
        CACHE_PATH.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy(DRIVE / CACHE_PATH.name, CACHE_PATH)
        print(f"Restored the cache from Drive: {DRIVE / CACHE_PATH.name}")
    if not CACHE_PATH.exists():
        raise SystemExit(
            f"No cache at {CACHE_PATH}, and none in {DRIVE}. Run Phase A, or point DRIVE_DIR at "
            "the folder holding the mirrored splices JSONL and turn LOAD_FROM_DRIVE on. The file "
            "name is keyed by model, corpus and window, so it must match this cell's RUN_TAG.")
    header, rows = load_splices(CACHE_PATH)
    print(f"Read {len(rows)} paragraphs / {cut_count(rows)} cuts back from {CACHE_PATH}")

# Provenance for the record. The cache header is the pre-registration of the run that actually
# produced these numbers -- model, dtype, BOS token, corpus -- so it wins over whatever this
# session has in memory; a session value only fills a gap an older header leaves.
RUN = {**(globals().get("PROVENANCE") or {}), **header}
RUN_MODEL_ID = RUN.get("model_id") or MODEL_ID
RUN_CORPUS = RUN.get("corpus") or globals().get("CORPUS_NAME") or "unknown"
if RUN_MODEL_ID != MODEL_ID:
    print(f"NOTE: this cache was scored with {RUN_MODEL_ID!r}, not the configured {MODEL_ID!r}; "
          "the record reports the cache's model.")

if UPLOAD_AUDIT and in_colab():
    from google.colab import files
    uploaded = files.upload()
    for name, data in uploaded.items():
        AUDIT_PATH.write_bytes(data)
        print(f"Saved the uploaded sheet as {AUDIT_PATH}")
elif (MIRROR_TO_DRIVE or LOAD_FROM_DRIVE) and (DRIVE / AUDIT_PATH.name).exists():
    shutil.copy(DRIVE / AUDIT_PATH.name, AUDIT_PATH)
    print(f"Restored the audit sheet from Drive: {DRIVE / AUDIT_PATH.name}")

audit = None
if AUDIT_PATH.exists():
    try:
        audit = load_audit_csv(AUDIT_PATH)
        print(f"Audit: {len(audit)} judgements loaded, {sum(1 for g, _ in audit.values() if not g)} failure(s).")
    except ValueError as exc:
        print(f"Audit sheet not filled in yet ({exc}).\nProceeding WITHOUT the audit; every cut carries audited=false.")
else:
    print("No audit sheet found; proceeding WITHOUT the audit (audited=false everywhere).")

record = analyse_9a(
    rows, header, audit=audit, model=RUN_MODEL_ID, seed=SEED, n_perm=N_PERM, n_boot=N_BOOT,
    min_per_group=MIN_PER_GROUP,
    notes=f"corpus={RUN_CORPUS}; dtype={RUN.get('dtype')}; bos={RUN.get('bos_token')!r}",
)
print()
print(f"{'stratum':<16}{'n close/far':>12}{'seg close/far':>16}{'median close':>14}{'median far':>12}{'ratio':>8}{'MW p':>8}")
for s in record["strata"]:
    print(f"{s['label']:<16}{str(s['n_close']) + '/' + str(s['n_far']):>12}"
          f"{str(s['median_seg_len_close']) + '/' + str(s['median_seg_len_far']):>16}"
          f"{s['median_close']:>14.1f}{s['median_far']:>12.1f}{s['ratio']:>8.3f}{s['p_mannwhitney']:>8.3f}")
bc = record["diagnostics"]["boundary_check"]
print(f"\nboundary check : {bc['n_endpoints_not_followed_by_a_sentence']}/{bc['n_endpoints_checked']} endpoints "
      f"not followed by a sentence start; {bc['n_cuts_affected']}/{bc['n_cuts']} cuts affected "
      f"({bc['by_pair']['close']['rate']:.1%} close, {bc['by_pair']['far']['rate']:.1%} far)")
if not bc["clean"]:
    for ex in bc["examples"][:3]:
        print(f"   {ex['paragraph']} @{ex['token']}: ...{ex['before']!r} || {ex['after']!r}...")

print()
print(verdict_9a(record))

## 12 · Figure

In [ ]:
#@title Draw fig_9a
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import Image, display

from m1_analyzer.experiments import experiment_figures as EF

plt.close(EF.fig_9a(record, path=str(FIG_PATH)))
display(Image(filename=str(FIG_PATH)))
print(FIG_PATH)

## 13 · Save and persist

The record is the experiment's single output: everything `fig_9a` needs plus the strata, the
stratified estimate, the pooled and continuous diagnostics, the audit outcome and the
pre-registration echoed back. Colab wipes `/content` when the runtime is recycled, so the Drive
cell copies the record, the figure, the corpus, the audit sheet and the cache to `MyDrive`.

In [ ]:
#@title Save the record
import json
import os
import tempfile

def atomic_write_json(path, payload):
    path = Path(path)
    fd, tmp = tempfile.mkstemp(dir=str(path.parent), prefix=f".{path.stem}.", suffix=".json")
    os.close(fd)
    Path(tmp).write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding="utf-8")
    os.replace(tmp, path)

atomic_write_json(RECORD_PATH, record)
print(f"record : {RECORD_PATH} ({RECORD_PATH.stat().st_size / 1024:.1f} KB)")
print(json.dumps({k: record[k] for k in ("stratified", "pooled", "audit", "meta")}, indent=2))

In [ ]:
#@title Persist results to Google Drive (survives a runtime disconnect)
import shutil

from m1_analyzer import in_colab

if in_colab():
    from google.colab import drive
    drive.mount("/content/drive")
    target = Path(DRIVE_DIR)
    target.mkdir(parents=True, exist_ok=True)
    for path in (RECORD_PATH, FIG_PATH, CORPUS_PATH, AUDIT_PATH, CACHE_PATH):
        if path.exists():
            shutil.copy(path, target / path.name)
            print("copied", target / path.name)
else:
    print("Not running in Colab -- skipping Drive mount.")

In [ ]:
#@title Download the record, the figure and the audit sheet to your machine
from m1_analyzer import in_colab

if in_colab():
    from google.colab import files
    for path in (RECORD_PATH, FIG_PATH, AUDIT_PATH):
        if path.exists():
            files.download(str(path))
else:
    print("Not in Colab; the files are already on disk under", OUT)

In [ ]:
#@title Run the test suite inside Colab
import subprocess, sys

result_proc = subprocess.run(
    [sys.executable, "-m", "pytest", "-q"], capture_output=True, text=True
)
print(result_proc.stdout[-4000:])
print(result_proc.stderr[-2000:], file=sys.stderr)

---

## Troubleshooting

| Symptom | Fix |
|---|---|
| `GITHUB_TOKEN is not set` / `401` / `403` / `cannot see <repo>` | See the bootstrap cell's message: add or fix the fine-grained PAT (Contents: Read on this repo), enable *Notebook access*, re-run cell 1. |
| `punkt sentence model is not installed` | The NLTK download failed (no network, or a proxy). Re-run the corpus cell; on a proxied machine set `NLTK_ALLOW_PROXIED_URLOPEN=1`. |
| The Wikipedia stream is slow or fails | `datasets` streams the dump from the Hub; a proxy may block it. Set `CORPUS` to a `.txt` or `.jsonl` file of your own paragraphs instead. |
| `No paragraph qualified` | The filter is too tight for the corpus: loosen `MIN_TOKENS` / `MAX_TOKENS` / `MIN_SENTENCES`. |
| `non-finite next-token state` | float16 overflowed inside the model. Set `DTYPE = "float32"`, `RESUME = False`, re-run phase A. |
| `CUDA out of memory` | Lower `BATCH_SIZE` (states are a full-vocabulary vector per position; a 150k vocabulary at 300 positions is 180 MB per sequence in float32) or `MAX_TOKENS`. |
| `was written for window=... but this run has ...` | The cache belongs to a run with a different pre-registration. Change `OUTPUT_DIR`, or set `RESUME = False` to rescore. |
| `was written for schema=1, but this run has schema=2` | The cache predates the two-sided boundary rule, so its endpoints were selected under a different definition. It cannot be resumed — set `RESUME = False` (or a new `OUTPUT_DIR`) and rescore. `analyse_9a` still *reads* such a cache and reports the contamination in `diagnostics.boundary_check`. |
| `UNDETERMINED: no length stratum has enough close and far cuts` | Score more paragraphs, or (recorded as a deviation) lower `MIN_PER_GROUP`. |
| `Audit sheet not filled in yet` | Fill the `grammatical` column with `y`/`n` for every row and put the file back at `AUDIT_PATH`, then re-run Phase B. |
| `NameError: name 'rows' is not defined` / a fresh runtime | Phase B reads the cache, not the model: leave the GPU cells alone, run cells 1–8 (bootstrap and configuration) so `CACHE_PATH` is defined, set `LOAD_FROM_DRIVE = True` with `DRIVE_DIR` pointing at the mirrored `splices_<RUN_TAG>.jsonl`, and run the Analyse cell. |
| `the text re-encodes to N tokens but the cache says M` | The surprisal cell was run with a model whose tokenizer is not the one the cache was scored with. Set `MODEL_ID` to the cache header's `model_id` (cell 4 prints `CACHE_PATH`; the header is its first line) and re-run cell 6, then this cell. |